# Lesson 1.5 — Agent SDK Hooks: Tool-Call Interception & Data Normalization

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Use PostToolUse hooks to normalize heterogeneous tool outputs before the model sees them
- Use interception hooks to block policy-violating tool calls and redirect to escalation
- Choose hooks over prompt-based enforcement when compliance must be guaranteed

**You will produce:** A PostToolUse normalizer and a pre-call interception hook that blocks refunds over $500.

## Concepts

**Hooks** let you intercept the agentic loop deterministically:

- **PostToolUse** hooks intercept *tool results* and transform them before the model processes them - e.g. normalizing Unix timestamps, ISO 8601, and numeric status codes from different MCP tools into one format so the agent reasons over consistent data.
- **Pre-call / tool-call interception** hooks inspect *outgoing tool calls* and can block policy violations (e.g. a refund exceeding $500) and redirect to an alternative workflow such as human escalation.

### Hooks vs prompts
Use hooks when business rules require **guaranteed** compliance. A prompt saying "never refund over $500" is probabilistic; a hook that rejects the call is deterministic. This mirrors the enforcement principle from Lesson 1.4.

> **Reference:** [Anthropic - Agent SDK hooks](https://docs.anthropic.com/en/docs/agents-and-tools/agent-sdk)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — PostToolUse: normalize heterogeneous timestamps

Three tools return three date formats. The hook emits one canonical ISO-8601 field.

In [ ]:
from datetime import datetime, timezone

def post_tool_use_normalize(tool_name, result):
    r = dict(result)
    ts = r.get("timestamp")
    if isinstance(ts, (int, float)):           # Unix seconds
        r["timestamp"] = datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()
    elif isinstance(ts, str) and ts.isdigit(): # numeric string
        r["timestamp"] = datetime.fromtimestamp(int(ts), tz=timezone.utc).isoformat()
    status_map = {0: "ok", 1: "pending", 2: "failed"}
    if isinstance(r.get("status"), int):
        r["status"] = status_map.get(r["status"], "unknown")
    return r

print(post_tool_use_normalize("lookup_order", {"timestamp": 1762880000, "status": 0}))
print(post_tool_use_normalize("get_customer", {"timestamp": "2026-11-11T00:00:00+00:00", "status": 1}))

## Code-Along: Step 2 — Interception: block refunds over a threshold

The hook inspects outgoing tool calls and redirects over-threshold refunds to escalation.

In [ ]:
REFUND_LIMIT = 500

def pre_tool_use_hook(tool_name, tool_input):
    if tool_name == "process_refund" and tool_input.get("amount", 0) > REFUND_LIMIT:
        return {"action": "block",
                "redirect": "escalate_to_human",
                "reason": f"Refund ${tool_input['amount']} exceeds ${REFUND_LIMIT} limit"}
    return {"action": "allow"}

print(pre_tool_use_hook("process_refund", {"amount": 750}))
print(pre_tool_use_hook("process_refund", {"amount": 200}))

## Code-Along: Step 3 — Wire a hook into the loop

Before executing any tool, consult the interception hook; after, run the normalizer.

In [ ]:
def guarded_execute(tool_name, tool_input, raw_result_fn):
    decision = pre_tool_use_hook(tool_name, tool_input)
    if decision["action"] == "block":
        return {"blocked": True, "redirect": decision["redirect"],
                "reason": decision["reason"]}
    raw = raw_result_fn(tool_name, tool_input)
    return post_tool_use_normalize(tool_name, raw)

demo = guarded_execute("process_refund", {"amount": 999}, lambda n, i: {})
print(demo)

## Exercise

Extend `pre_tool_use_hook` to also block `process_refund` when the session is unverified (no customer ID), reusing the gate idea from Lesson 1.4. Then add a PostToolUse rule that trims an order-lookup result from 40 fields down to the 5 return-relevant ones (preview of Lesson 5.1 context trimming).

In [ ]:
# TODO: add an unverified-session block branch to pre_tool_use_hook
# TODO: add a PostToolUse trim keeping only {order_id, status, total, date, items}

## Key Takeaways

- PostToolUse hooks normalize heterogeneous tool outputs before the model reasons over them.
- Interception hooks block policy-violating calls and redirect (e.g. to escalation).
- Choose hooks over prompts whenever compliance must be guaranteed, not merely likely.
- Hooks are the deterministic counterpart to the prerequisite gates from Lesson 1.4.

---

**Next:** [Lesson 1.6 — Task Decomposition & Session State](./lesson-1.6-decomposition-and-sessions.ipynb)